# 02 - The Intern (Fine-Tuning - Unsloth Optimized)

## Setup (Colab)

In [ ]:
# Install Unsloth & Dependencies
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q python-dotenv datasets

import os
repo_name = "zuucrew-ai-aee-mini-project-1"
repo_url = f"https://github.com/Sulamaxx/{repo_name}.git"

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(f"/content/{repo_name}"):
        print(f"Cloning {repo_name}...")
        !git clone {repo_url}
    
    os.chdir(f"/content/{repo_name}")
    print(f"Switched to repo directory: {os.getcwd()}")

## Configuration & Imports

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from dotenv import load_dotenv
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"

load_dotenv()

device = "cuda" if torch.cuda.is_available() else "cpu"
max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

train_file = Path("artifacts/data_factory/train.jsonl")
test_file = Path("artifacts/data_factory/golden_test_set.jsonl")
output_dir = Path("artifacts/intern_adapters")

print(f"Current Dir: {os.getcwd()}")
print(f"Device: {device}")

## Loading the Dataset

In [ ]:
if not train_file.exists():
    raise FileNotFoundError(
        f"{train_file} not found! Current Dir: {os.getcwd()}"
    )

dataset = load_dataset("json", data_files={"train": str(train_file), "test": str(test_file)})
print(f"Train size: {len(dataset['train'])}")

## Model and LoRA Initialization

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("Unsloth model and LoRA adapters ready.")

## Training with SFTTrainer

In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruction}\n{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts, }

dataset["train"] = dataset["train"].map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 250, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer.train()

## Inference Pipeline

In [ ]:
FastLanguageModel.for_inference(model)
model.save_pretrained(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

def query_intern(question):
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 256, 
        use_cache = True,
        repetition_penalty = 1.2 
    )
    response = tokenizer.batch_decode(outputs)
    return response[0].split("assistant<|end_header_id|>\n\n")[-1].replace("<|eot_id|>", "").strip()

sample_q = "What were Uber's total revenues in 2024?"
print(f"Question: {sample_q}")
print(f"Answer: {query_intern(sample_q)}")

In [ ]:
def save_to_drive():
    try:
        from google.colab import drive
        print("Requesting Drive access...")
        drive.mount('/content/drive')
        
        import os
        save_path = "/content/drive/MyDrive/AIEngineering/Project1/intern_adapters"
        os.makedirs(save_path, exist_ok=True)
        
        print(f"Backing up adapters to {save_path}...")
        import shutil
        source_dir = "artifacts/intern_adapters/"
        for file_name in os.listdir(source_dir):
            shutil.copy(os.path.join(source_dir, file_name), save_path)
            
        print(f"Adapters backed up successfully!")
    except Exception as e:
        print(f"Mount failed: {e}")

save_to_drive()